In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq
import json
load_dotenv()

for _p in [Path.cwd(), *Path.cwd().resolve().parents]:
    if (_p / "app" / "db" / "database.py").exists():
        sys.path.insert(0, str(_p))
        break
    if (_p / "calender_agent" / "app" / "db" / "database.py").exists():
        sys.path.insert(0, str(_p / "calender_agent"))
        break
else:
    sys.path.insert(0, str(Path.cwd().resolve().parents[2] if len(Path.cwd().resolve().parents) >= 3 else Path.cwd()))

from app.db.database import SessionLocal
from app.tools.calender import create_calender_event

In [ ]:
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print(os.getenv("GROQ_API_KEY") is not None)
v = os.getenv("GROQ_API_KEY") or ""
print(repr(v[:5]), repr(v[-4:]), len(v))
messages=[]

create_calender_event={
    "type" : "function",
    "function" : {
        "name" : "create_calender_event",
        "description" : "create a calender event on a particular data at a particular time",
        "parameters" : {
            "title": {
                "type": "string",
                "description": "Title of the calendar event."
            },
            "date": {
                "type": "string",
                "description": "Calendar date in YYYY-MM-DD format."
            },
            "time": {
                "type": "string",
                "description": "Event time in HH:MM 24-hour format."
            },
            "required": ["title", "date", "time"],
            "additionalProperties": False
        }
    }
}

list_calender_event={
    "type":"function",
    "function":{
        "name" : "list_calender_event",
        "description" : "list the calender events on a particular date",
        "parameters" : {
            "date" : {
                "type" : "string",
                "description" : "Calendar date in YYYY-MM-DD format."
            },
            "required": ["date"],
            "additionalProperties": False
        }
    }
}


tool_registery={
    "create_calender_event" : create_calender_event,
    "list_calender_event" : list_calender_event
}

True
'hf_on' 'wBem' 37


In [ ]:
def args_validator(args : dict , tool_schema : dict):
    required_args=set(tool_schema["parameters"]["required"])
    provided_args=set(args.keys())
    allowed_args = set(tool_schema["parameters"]["properties"].keys())

    missing_args=required_args-provided_args
    additional_args=provided_args-allowed_args

    if missing_args:
        return {
            "ok" : False,
            "error" : {
                "code" : "MISSING ARGUMENTS",
                "message" : f"expected {missing_args}"
            }
        }

    if additional_args:
        return {
            "ok" : False,
            "error" : {
                "code" : "INVALID ARGUMENTS",
                "message" : f"{additional_args} are not required"
            }
        }

    return {
        "ok": True
    }


def tool_executor(tool_name,tool_args):
    db=SessionLocal()
    try:
        tool=tool_registery[tool_name]
        tool_res=tool(**tool_args,db=db)
        return tool_res
    
    finally:
        db.close()

In [ ]:
messages.append({
    "role":"system",
    "content" : "You are a helpful AI assistant and your task is to answer questions without hallucinations."
})

available_tools=[create_calender_event,list_calender_event]

user_input=input(">>")
print(f">> {user_input}")

messages.append(
    {
        "role" : "user",
        "content" : user_input
    }
)

while True:
    response=client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        tools=available_tools
    )

    assistant_response=response.choices[0].message
    
    if assistant_response.tool_calls:
        tool_call_list=[]
        for tool_call in assistant_response.tool_calls:
            tool_name=tool_call.function.name
            tool_args=tool_call.function.arguments

            tool_call_list.append({
                "id" : tool_call.id,
                "type" : "function",
                "function" : {
                    "name" : tool_name,
                    "arguments" : tool_args
                }
            })

        messages.append({
            "role":"assistant",
            "content" : None,
            "tool_calls" : tool_call_list
        })


        for tool_call in assistant_response.tool_calls:
                
            tool_name=tool_call.function.name
            tool_args=tool_call.function.arguments

            if tool_name in tool_registery:
                try:
                    args=json.loads(tool_args)
                    
                except json.JSONDecodeError:
                    tool_res={
                        "ok" : False,
                        "error" : {
                            "code" : "INVALID_JSON",
                            "message": "Tool arguments must be valid JSON."
                        }
                    }
                else : 
                    if isinstance(args,dict):
                        args_valid=args_validator(args,tool_registery[tool_name]["function"])
                        if args_valid["ok"]:
                            tool_res=args_valid
                        else:    
                            tool_res=tool_executor(tool_name,tool_args)
                        
                    else : 
                        tool_res={
                            "ok" : False,
                            "error" : {
                                "code" : "INVALID_DATA_FORMAT",
                                "message" : "expected dictionary"
                            }
                        }

            else :
                tool_res = {
                    "ok": False,
                    "error": {
                        "code": "UNKNOWN_TOOL",
                        "message": f"Tool '{tool_name}' is not available."
                    }
                }

            messages.append({
                "role" : "tool",
                "content" : json.dumps(tool_res),
                "tool_call_id" : tool_call.id
            })

    else:
        print(f"LLM >> {assistant_response.content}")
        messages.append({
            "role" : "assistant",
            "content" : assistant_response.content
        })

        user_ip=input(f">> ")
        if user_ip.lower() is "exit":
            break

        messages.append({
            "role" : "user",
            "content" : user_ip
        })

<>:73: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:73: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?


>> exit


/tmp/ipykernel_1208/2927881414.py:73: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if user_ip.lower() is "exit":


TypeError: create_calender_event() got an unexpected keyword argument 'date'

In [ ]:
# Next objectives


# Duplicate calls

# Error classification- validation error
# - tool/application error
# - transient error
# - infrastructure error

# Retry policy- what errors are retryable?
# - max retries
# - exponential backoff

# Multiple tool calls- sequential execution
# - dependencies between calls

# Parallel tool calls- when calls are independent
# - concurrency problems

# Timeouts / cancellation

# Idempotency & side effects

# Tool permissions- which agent/user can invoke which tool

# Basic observability- tool name
# - arguments
# - latency
# - success/failure
# - error

# Evaluation- did the model select the right tool?
# - were arguments correct?
# - did execution produce the expected result?

SyntaxError: invalid syntax (1260018186.py, line 1)